# Chapter 2: Select, From & Where

### Core Concepts:
* **`SELECT`**: Pulls specific columns.
* **`FROM`**: Identifies the dataset/table (`bigquery-public-data.openaq.global_air_quality`).
* **`WHERE`**: Filters results by rows matching a condition.
* **`QueryJobConfig`**: Controls execution parameters like query size dry-runs and limit rules.

In [13]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client()

## Task 1: View Table Schema

Let's look at the schema of the `global_air_quality` table inside the `openaq` dataset to understand what columns are available.

In [14]:
dataset_ref = client.dataset("openaq", project="bigquery-public-data")
table_ref = dataset_ref.table("global_air_quality")
table = client.get_table(table_ref)

for field in table.schema:
    print(f"{field.name:<15} | {field.field_type:<10} | {field.description}")

location        | STRING     | None
city            | STRING     | None
country         | STRING     | None
pollutant       | STRING     | None
value           | FLOAT      | None
timestamp       | TIMESTAMP  | None
unit            | STRING     | None
source_name     | STRING     | None
latitude        | FLOAT      | None
longitude       | FLOAT      | None
averaged_over_in_hours | FLOAT      | None
location_geom   | GEOGRAPHY  | None


## Task 2: Write your first SQL Query

Open the file `queries/cities.sql` and write a query that selects the `city` column from this table for all rows where the `country` column is `'US'`.

Once you have written it, run the block below to load and inspect it.

In [18]:
# Load query from local SQL file
with open("queries/cities.sql", "r") as file:
    query_cities = file.read()

print("Query Loaded:")
print(query_cities)

Query Loaded:
-- Write your SQL query here
-- Select the 'city' column
-- From the table `bigquery-public-data.openaq.global_air_quality`
-- Where the 'country' column is 'US'

SELECT city
FROM `bigquery-public-data.openaq.global_air_quality`
WHERE country = 'US';



## Task 3: Perform a Query Dry Run

Before executing the query, we should always check how much data it scans to avoid unexpected costs.

In [16]:
# Configure a dry run
dry_run_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run_job = client.query(query_cities, job_config=dry_run_config)

print(f"This query will scan: {dry_run_job.total_bytes_processed / (1024**2):.2f} MB of data.")

This query will scan: 91.38 MB of data.


## Task 4: Execute query with a Cost Limit

Let's execute the query. To protect ourselves, we'll set `maximum_bytes_billed` to 10 MB (10,485,760 bytes) which is more than enough for this tiny query, but will block any massive accidental scans.

In [19]:
# Limit maximum bytes billed to 10 MB
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=100 * 1024 * 1024)

try:
    query_job = client.query(query_cities, job_config=safe_config)
    df_cities = query_job.to_dataframe()
    print(f"Query returned {len(df_cities)} rows.")
    print(df_cities.head())
except Exception as e:
    print(f"Query failed/blocked: {e}")

/Users/danwooster/1. DEV/DS-AI/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Query returned 1421351 rows.
     city
0  HOWARD
1  HOWARD
2  HOWARD
3  HOWARD
4  HOWARD
